# Train Anomaly Detection Model

In [ ]:
!pip install --upgrade pip
!pip install onnx==1.17.0 onnxruntime==1.19.2 tf2onnx==1.16.1
!pip install --upgrade "protobuf==5.28.3"
!pip install tensorflow scikit-learn pandas numpy

In [ ]:
import numpy as np
import pandas as pd
import datetime
from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization, Activation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import class_weight
import tf2onnx
import onnx
import pickle
from pathlib import Path

In [ ]:
feature_indexes = [0, 1, 2, 3, 4, 5, 6, 7, 8]
label_indexes = [9]

df = pd.read_csv('data/train.csv')
X_train = df.iloc[:, feature_indexes].values
y_train = df.iloc[:, label_indexes].values

df = pd.read_csv('data/validate.csv')
X_val = df.iloc[:, feature_indexes].values
y_val = df.iloc[:, label_indexes].values

df = pd.read_csv('data/test.csv')
X_test = df.iloc[:, feature_indexes].values
y_test = df.iloc[:, label_indexes].values

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

Path("artifact").mkdir(parents=True, exist_ok=True)
with open("artifact/test_data.pkl", "wb") as handle:
    pickle.dump((X_test, y_test), handle)
with open("artifact/scaler.pkl", "wb") as handle:
    pickle.dump(scaler, handle)

class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train.ravel()
)
class_weights = {i : class_weights[i] for i in range(len(class_weights))}

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Test samples: {len(X_test)}")
print(f"Class weights: {class_weights}")

In [ ]:
model = Sequential()
model.add(Dense(32, activation='relu', input_dim=len(feature_indexes)))
model.add(Dropout(0.2))
model.add(Dense(32))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Dense(32))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
import time

start = time.time()
epochs = 30

history = model.fit(
    X_train,
    y_train,
    epochs=epochs,
    validation_data=(X_val, y_val),
    verbose=True,
    class_weight=class_weights
)

end = time.time()
print(f"\nTraining complete. Time: {end-start:.2f} seconds")

In [ ]:
import tensorflow as tf
import os

@tf.function(input_signature=[tf.TensorSpec([None, X_train.shape[1]], tf.float32, name='dense_input')])
def model_fn(x):
    return model(x)

model_proto, _ = tf2onnx.convert.from_function(
    model_fn,
    input_signature=[tf.TensorSpec([None, X_train.shape[1]], tf.float32, name='dense_input')]
)

os.makedirs("models/anomaly/1", exist_ok=True)
onnx.save(model_proto, "models/anomaly/1/model.onnx")

print("Model saved successfully!")

In [ ]:
!ls -alRh ./models/

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score
import onnxruntime as rt

with open('artifact/scaler.pkl', 'rb') as handle:
    scaler = pickle.load(handle)
with open('artifact/test_data.pkl', 'rb') as handle:
    (X_test, y_test) = pickle.load(handle)

In [ ]:
sess = rt.InferenceSession("models/anomaly/1/model.onnx", providers=rt.get_available_providers())
input_name = sess.get_inputs()[0].name
output_name = sess.get_outputs()[0].name

y_pred_temp = sess.run([output_name], {input_name: X_test.astype(np.float32)})
y_pred_temp = np.asarray(np.squeeze(y_pred_temp[0]))

threshold = 0.5
y_pred = np.where(y_pred_temp > threshold, 1, 0)

In [ ]:
y_test_arr = y_test.squeeze()
correct = np.equal(y_pred, y_test_arr).sum().item()
acc = (correct / len(y_pred)) * 100
precision = precision_score(y_test_arr, np.round(y_pred))
recall = recall_score(y_test_arr, np.round(y_pred))

print(f"Evaluation Metrics:")
print(f"  Accuracy: {acc:.2f}%")
print(f"  Precision: {precision:.4f}")
print(f"  Recall: {recall:.4f}")

c_matrix = confusion_matrix(y_test_arr, y_pred)
ConfusionMatrixDisplay(c_matrix).plot()

In [ ]:
normal_activity = [[192, 168, 10, 15, 10, 0, 0, 0, 0]]

prediction = sess.run([output_name], {input_name: scaler.transform(normal_activity).astype(np.float32)})
prob = np.squeeze(prediction)

print("Normal activity test:")
print(f"  Anomaly probability: {prob:.4f}")
print(f"  Predicted as anomaly: {prob > threshold}")

suspicious_activity = [[203, 0, 113, 77, 3, 1, 0, 3, 1]]

prediction = sess.run([output_name], {input_name: scaler.transform(suspicious_activity).astype(np.float32)})
prob = np.squeeze(prediction)

print("\nSuspicious activity test:")
print(f"  Anomaly probability: {prob:.4f}")
print(f"  Predicted as anomaly: {prob > threshold}")